## Application in Marketing

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict
import math

# Transaction data: (user, product) 交易資料 （用戶 商品）
# Edge list
transactions =[
    # Original core behavior 
    ('U1', 'P1'), ('U1', 'P2'), ('U1', 'P3'), # U 1 購買 p1, p2, p3
    ('U2', 'P1'), ('U2', 'P2'),
    ('U3', 'P2'), ('U3', 'P4'),
    ('U4', 'P3'), ('U4', 'P4'),
    ('U5', 'P1'), ('U5', 'P4'),
    ('U6', 'P1'), ('U6', 'P2'), ('U6', 'P5'),  
    ('U7', 'P5'), ('U7', 'P6'),                
    ('U8', 'P2'), ('U8', 'P4'), ('U8', 'P7'),  
    ('U9', 'P6'), ('U9', 'P7'),                
    ('U10', 'P1'), ('U10', 'P3'), ('U10', 'P5'), ('U10', 'P7'),  
]
# step II: put into dataframe 放入dataframe 
df = pd.DataFrame(transactions, columns=['user', 'product'])

# Build user-product bipartite graph 建立用戶-商品二分網
B = nx.Graph()
B.add_nodes_from(df['user'].unique(), bipartite=0)  # users 用戶節點
B.add_nodes_from(df['product'].unique(), bipartite=1)  # products 商品節點
B.add_edges_from(transactions)

# get neighbors of a product (i.e., users who bought it) 工具:那些用戶 買該商品
def get_users(product):
    return set(B.neighbors(product))

# get products of a user 工具:用戶買那些商品
def get_products(user):
    return set(B.neighbors(user))

In [ ]:
# Step 1: Segment users by activity (e.g., # of purchases) 依購買活動來區分用戶
user_activity = df.groupby('user').size()
high_value = user_activity[user_activity >= 2].index  # Active users 用戶節點 購買兩次以上才計入討論

# Step 2: Filter transactions from raw dataframe for high-value users 從原始的資料裡找出 該用戶的交易資料
df_hv = df[df['user'].isin(high_value)]

# Rebuild bipartite graph for this segment 依該區分建立二分網絡
B_hv = nx.Graph()
B_hv.add_nodes_from(df_hv['user'].unique(), bipartite=0)
B_hv.add_nodes_from(df_hv['product'].unique(), bipartite=1)
B_hv.add_edges_from(df_hv.values)

# Step 3: Compute Jaccard for P1 in high-value segment 二分網絡裡 以P1為基礎 計算跟其他商品的 Jaccard 
def jaccard_in_graph(G, p1, p2):
    N1 = set(G.neighbors(p1))
    N2 = set(G.neighbors(p2))
    if not N1 or not N2:
        return 0
    return len(N1 & N2) / len(N1 | N2)

products = df_hv['product'].unique()
scores = [(p, jaccard_in_graph(B_hv, 'P1', p)) for p in products if p != 'P1']
print("High-value segment affinities for P1:", sorted(scores, key=lambda x: -x[1]))

In [ ]:
# Assume all products a user ever bought = "considered" 把用戶所有購買過的商品都加以考慮
# (In real life, use view logs)

def adamic_adar_score(p1, p2, graph): # 計算AA分數
    common_users = set(graph.neighbors(p1)) & set(graph.neighbors(p2))
    score = 0.0
    for u in common_users:
        deg = len(list(graph.neighbors(u)))  # # products user considered
        if deg > 1:
            score += 1 / math.log(deg)
    return score

# Compare P1 and P2 (often bought together → complement) 比較P1 和P2 通常一起購買會被當作"互補商品“
# vs P1 and P4 (never bought together, but U2/U5 saw both → substitute?) 比較P1 和P4 未曾一起購買 但是u2/u5

print("AA(P1,P2):", adamic_adar_score('P1', 'P2', B))  # Higher if same users consider both 如果用戶同時購買--> AA會高
print("AA(P1,P4):", adamic_adar_score('P1', 'P4', B))  # May be non-zero even if not co-purchased 並非同時購買 也不會為零

# Complement signal: Jaccard on purchase
print("Jaccard(P1,P2):", jaccard_in_graph(B, 'P1', 'P2'))  # High → complement 高--> 互補商品 
print("Jaccard(P1,P4):", jaccard_in_graph(B, 'P1', 'P4'))  # Low → not complement 低--> 沒有互補關係 

In [ ]:
# Suppose P5 is in same category as P1 and P2  假設p5和p1 and p2是相同類別的商品
similar_products = ['P1', 'P2']

# Estimate affinity of P5 to P3 as average of P1→P3 and P2→P3 利用 相同類別的 p1 and p2 (加權) 平均出 p3的親和估計值
target = 'P3'
estimated_jaccard = np.mean([
    jaccard_in_graph(B, sp, target) for sp in similar_products
])
print(f"Estimated Jaccard(P5, {target}): {estimated_jaccard:.2f}")

In [ ]:
# Build product co-purchase graph (one-mode projection) 進行單模投影
product_graph = nx.Graph()
products = df['product'].unique()
product_graph.add_nodes_from(products)

# Add edges weighted by co-purchase count 把共同購買的次數當作是 邊的權數
cooc = defaultdict(int)
for user in df['user'].unique():
    prods = sorted(get_products(user))
    for i in range(len(prods)):
        for j in range(i+1, len(prods)):
            cooc[(prods[i], prods[j])] += 1

for (p1, p2), w in cooc.items():
    product_graph.add_edge(p1, p2, weight=w)

# Degree centrality = sum of co-purchase counts 度中心性= 共同購買次數 的加總 
centrality = {p: sum(d['weight'] for _, d in product_graph[p].items()) 
              for p in product_graph.nodes()}

# Basket size when product is bought 計算商品被購買時 共有幾項商品
basket_size = df.groupby('product')['user'].apply(
    lambda x: df[df['user'].isin(x)]['product'].groupby(df['user']).size().mean()
).to_dict()

print("Product | Centrality | Avg Basket Size")
for p in products:
    print(f"{p:6} | {centrality.get(p,0):10} | {basket_size.get(p,0):.2f}")

In [ ]:
# Define sensitive categories 定義敏感商品類別 
sensitive_keywords = {'diet', 'laxative', 'detox'} # 節食 通便劑 戒癮
product_names = {
    'P1': 'multivitamin', # 維他命
    'P2': 'diet tea',     # 節食茶
    'P3': 'protein bar',  # 蛋白質能量棒
    'P4': 'laxative supplement', # 通便劑
    'P5': 'resistance band',  # 彈力帶
    'P6': 'yoga mat',         # 瑜珈墊
    'P7': 'smart water bottle' # 智慧型水瓶
}

def is_sensitive(pid):
    return any(kw in product_names[pid].lower() for kw in sensitive_keywords)

# Find high-affinity pairs involving sensitive products 找出敏感商品的高親和商品搭配
risky_pairs = []
for (p1, p2), count in cooc.items():
    if count >= 1 and is_sensitive(p1) and is_sensitive(p2):
        risky_pairs.append((p1, p2, count))

print("Risky pairs to suppress:", risky_pairs)

## Application in Social Media Analysis

In [ ]:
import pandas as pd
import networkx as nx
import math
from collections import defaultdict

# Social media data: (user, hashtag) 用戶 標籤
# ClimateAction, Renewables, GreenNewDeal, FossilFuels, Jobs, Innovation
# 氣候行動, 再生能源, 綠色新政, 石化燃料, 就業, 創新
posts = [
    ('Alice', '#ClimateAction'), ('Alice', '#Renewables'),
    ('Bob', '#ClimateAction'), ('Bob', '#GreenNewDeal'),
    ('Carol', '#GreenNewDeal'), ('Carol', '#Policy'),
    ('Dave', '#FossilFuels'), ('Dave', '#Jobs'),
    ('Eve', '#Renewables'), ('Eve', '#Innovation'),
]

df = pd.DataFrame(posts, columns=['user', 'hashtag'])

# Build bipartite graph: users (mode 0), hashtags (mode 1) 建立二分網絡
B = nx.Graph()
B.add_nodes_from(df['user'].unique(), bipartite=0)
B.add_nodes_from(df['hashtag'].unique(), bipartite=1)
B.add_edges_from(posts)

def get_users(hashtag):
    return set(B.neighbors(hashtag))

def get_hashtags(user):
    return set(B.neighbors(user))

In [ ]:
def jaccard_hashtags(h1, h2): # 計算jaccard
    U1, U2 = get_users(h1), get_users(h2)
    if not U1 or not U2:
        return 0
    return len(U1 & U2) / len(U1 | U2)

hashtags = df['hashtag'].unique()
affinity = {}
for i, h1 in enumerate(hashtags):
    for h2 in hashtags[i+1:]:
        score = jaccard_hashtags(h1, h2)
        if score > 0.1:  # threshold
            affinity[(h1, h2)] = score

print("Strong hashtag affinities:") # 標籤- 標籤 親和力
for (h1, h2), s in sorted(affinity.items(), key=lambda x: -x[1]):
    print(f"{h1} — {h2}: {s:.2f}")

In [ ]:
# Build user co-affiliation graph 單模投影而成 用戶網絡
user_graph = nx.Graph()
user_graph.add_nodes_from(df['user'].unique())

# Add edges if users share ≥1 hashtag 如果用戶的標籤數量大於1
for u1 in df['user'].unique():
    for u2 in df['user'].unique():
        if u1 >= u2:
            continue
        shared = get_hashtags(u1) & get_hashtags(u2)
        if shared:
            user_graph.add_edge(u1, u2, weight=len(shared))

# Find bridge users 利用介數中心性 找橋接用戶
betweenness = nx.betweenness_centrality(user_graph, weight='weight')
print("Bridge users (high betweenness):")
for user, score in sorted(betweenness.items(), key=lambda x: -x[1]):
    print(f"{user}: {score:.3f}")

In [ ]:
def adamic_adar_hashtags(h1, h2): # 計算AA
    common_users = get_users(h1) & get_users(h2)
    score = 0.0
    for u in common_users:
        total_tags = len(get_hashtags(u))  # how "focused" is this user?
        if total_tags > 1:
            score += 1 / math.log(total_tags)
    return score

# Compare two pairs 兩標籤的相似度
print("AA(#ClimateAction, #GreenNewDeal):", adamic_adar_hashtags('#ClimateAction', '#GreenNewDeal'))
print("AA(#ClimateAction, #Jobs):", adamic_adar_hashtags('#ClimateAction', '#Jobs'))  # likely 0

In [ ]:
brand_hashtag = '#GreenFuture'  # 假設目標標籤 綠色未來
risky_hashtags = ['#Scam', '#Pollution', '#Boycott'] # 風險標籤: 欺詐,污染,抵制

# Simulate: add a risky post 多增加一位用戶 Zoe
df_risk = df.copy()
df_risk = pd.concat([df_risk, pd.DataFrame([('Zoe', '#GreenFuture'), ('Zoe', '#Pollution')], columns=['user','hashtag'])])

# Rebuild graph 重建二分網絡
B_risk = nx.Graph()
B_risk.add_nodes_from(df_risk['user'].unique(), bipartite=0)
B_risk.add_nodes_from(df_risk['hashtag'].unique(), bipartite=1)
B_risk.add_edges_from(df_risk.values)

def jaccard_in_graph(G, h1, h2):
    U1, U2 = set(G.neighbors(h1)), set(G.neighbors(h2))
    return len(U1 & U2) / len(U1 | U2) if (U1 | U2) else 0

for risky in risky_hashtags:
    if risky in B_risk:
        sim = jaccard_in_graph(B_risk, brand_hashtag, risky)
        if sim > 0:
            print(f"⚠️  Brand risk: {brand_hashtag} co-occurs with {risky} (Jaccard={sim:.2f})")

In [ ]:
# Find hashtag pairs used by ≥2 users but with low total reach (niche) 
# 找標籤配對 被兩位以上的用戶所選上的 
niche_pairs = []
for i, h1 in enumerate(hashtags):
    for h2 in hashtags[i+1:]:
        common = get_users(h1) & get_users(h2)
        if len(common) >= 2 and len(get_users(h1)) <= 3 and len(get_users(h2)) <= 3:
            niche_pairs.append((h1, h2, common))

print("Suspicious niche co-use:")
for h1, h2, users in niche_pairs:
    print(f"{h1}, {h2} → used by {users}")

## Application in Finance

In [ ]:
import pandas as pd
import networkx as nx
import math
import numpy as np

# Sample portfolio holdings (investor, asset) 投資客 資產
holdings = [
    ('Vanguard', 'AAPL'), ('Vanguard', 'MSFT'),
    ('ARK', 'TSLA'), ('ARK', 'ROKU'),
    ('Fidelity', 'AAPL'), ('Fidelity', 'TSLA'), ('Fidelity', 'NVDA'),
    ('Bridgewater', 'GLD'), ('Bridgewater', 'TLT'),
    ('Renaissance', 'NVDA'), ('Renaissance', 'TSLA')
]

df = pd.DataFrame(holdings, columns=['investor', 'asset'])

# Build bipartite graph 建立二分網絡
B = nx.Graph()
B.add_nodes_from(df['investor'].unique(), bipartite=0)   # investors
B.add_nodes_from(df['asset'].unique(), bipartite=1)      # assets
B.add_edges_from(holdings)

def get_investors(asset):
    return set(B.neighbors(asset))

def get_assets(investor):
    return set(B.neighbors(investor))

In [ ]:
def jaccard_assets(a1, a2):
    I1, I2 = get_investors(a1), get_investors(a2)
    if not I1 or not I2:
        return 0
    return len(I1 & I2) / len(I1 | I2)

assets = df['asset'].unique()
print("Asset affinities (Jaccard):") # 資產親和
for i, a1 in enumerate(assets):
    for a2 in assets[i+1:]:
        score = jaccard_assets(a1, a2)
        if score > 0.1:
            print(f"{a1} — {a2}: {score:.2f}")

In [ ]:
def common_holders(a1, a2): # 共同持有的投資客
    return len(get_investors(a1) & get_investors(a2))

print("High co-holding pairs (contagion risk):")
for i, a1 in enumerate(assets):
    for a2 in assets[i+1:]:
        cn = common_holders(a1, a2)
        if cn >= 2:
            print(f"{a1} & {a2}: held by {cn} same investors")

In [ ]:
def adamic_adar_assets(a1, a2): #計算AA
    common_inv = get_investors(a1) & get_investors(a2)
    score = 0.0
    for inv in common_inv:
        portfolio_size = len(get_assets(inv))  # how concentrated is this investor?
        if portfolio_size > 1:
            score += 1 / math.log(portfolio_size)
    return score

print("AA scores (focus-weighted affinity):") 
for i, a1 in enumerate(assets):
    for a2 in assets[i+1:]:
        aa = adamic_adar_assets(a1, a2)
        if aa > 0.5:
            print(f"{a1} — {a2}: AA = {aa:.2f}")

In [ ]:
# Suppose we compare Q1 vs Q2 holdings
holdings_q1 = [('Vanguard','AAPL'), ('ARK','TSLA')]
holdings_q2 = holdings  # assume Q2 has more overlap

df_q2 = pd.DataFrame(holdings_q2, columns=['investor','asset'])
B_q2 = nx.Graph()
B_q2.add_nodes_from(df_q2['investor'].unique(), bipartite=0)
B_q2.add_nodes_from(df_q2['asset'].unique(), bipartite=1)
B_q2.add_edges_from(holdings_q2)

def jaccard_in_graph(G, a1, a2):
    I1, I2 = set(G.neighbors(a1)), set(G.neighbors(a2))
    return len(I1 & I2) / len(I1 | I2) if (I1 | I2) else 0

# Compare TSLA-NVDA affinity
j_q1 = 0  # not co-held in Q1
j_q2 = jaccard_in_graph(B_q2, 'TSLA', 'NVDA')
if j_q2 - j_q1 > 0.5:
    print(" Herding signal: TSLA & NVDA co-holding surged")

In [ ]:
activist_funds = {'ARK', 'Renaissance'}

def is_activist_co_held(a1, a2):
    common = get_investors(a1) & get_investors(a2)
    return len(common & activist_funds) >= 1 and len(common) <= 2

print("Potential stealth pairs:")
for i, a1 in enumerate(assets):
    for a2 in assets[i+1:]:
        if is_activist_co_held(a1, a2):
            print(f"{a1} & {a2} co-held by activist(s): {get_investors(a1) & get_investors(a2)}")